# 🎯 6. Iterators & Iterables

Understanding the **iterator protocol** explains how `for` loops, generators, and many built-in functions work under the hood.

| ☕ Java | 🐍 Python |
|---------|----------|
| `Iterable<T>` interface → `iterator()` | `__iter__()` method |
| `Iterator<T>` interface → `hasNext()` + `next()` | `__next__()` + `StopIteration` |
| Enhanced for-loop uses `Iterable` | `for` loop uses `__iter__` protocol |

## 6.1 Iterable vs Iterator

| Concept | Has | Role |
|---------|-----|------|
| **Iterable** | `__iter__()` | Can create an iterator |
| **Iterator** | `__iter__()` + `__next__()` | Produces values one by one |

Lists, tuples, strings, dicts are **iterables** — the `for` loop calls `iter()` on them to get an iterator.

In [ ]:
# A list is an iterable — you can get an iterator from it
my_list = [10, 20, 30]

# What 'for x in my_list' does under the hood:
iterator = iter(my_list)         # Calls my_list.__iter__()
print(next(iterator))            # 10 — calls iterator.__next__()
print(next(iterator))            # 20
print(next(iterator))            # 30

try:
    next(iterator)               # StopIteration!
except StopIteration:
    print("Iterator exhausted")

# Important: list is reusable, iterator is NOT
print(f"\nList (reusable): {list(my_list)} then {list(my_list)}")
it = iter(my_list)
print(f"Iterator (single-use): {list(it)} then {list(it)}")

## 6.2 Custom Iterator — Class-based

Implement `__iter__` and `__next__` to make any object iterable.

In [ ]:
class CountDown:
    """Iterator that counts down from n to 1."""
    
    def __init__(self, start: int):
        self.current = start
    
    def __iter__(self):
        """Return self — this object IS the iterator."""
        return self
    
    def __next__(self) -> int:
        """Return next value or raise StopIteration."""
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value

# Works in for loops
for num in CountDown(5):
    print(f"  {num}", end=" ")
print()

# Works with built-in functions
print(f"Sum: {sum(CountDown(10))}")
print(f"List: {list(CountDown(5))}")

## 6.3 Separate Iterable and Iterator (Reusable)

**Problem:** When `__iter__` returns `self`, the object is single-use.
**Solution:** Separate the iterable (factory) from the iterator (state).

In [ ]:
class Range:
    """Reusable iterable — creates a fresh iterator each time."""
    
    def __init__(self, start: int, end: int):
        self.start = start
        self.end = end
    
    def __iter__(self):
        """Return a NEW iterator each time."""
        return RangeIterator(self.start, self.end)

class RangeIterator:
    """Iterator — tracks position."""
    
    def __init__(self, start: int, end: int):
        self.current = start
        self.end = end
    
    def __iter__(self):
        return self
    
    def __next__(self) -> int:
        if self.current >= self.end:
            raise StopIteration
        value = self.current
        self.current += 1
        return value

my_range = Range(1, 6)

# Can iterate multiple times!
print(f"First pass:  {list(my_range)}")
print(f"Second pass: {list(my_range)}")

## 6.4 `__getitem__` Fallback Protocol

Python also supports iteration via `__getitem__` (index-based).
If a class has `__getitem__` but no `__iter__`, Python will still allow `for` loops.

In [ ]:
class OldStyleSequence:
    """No __iter__, but __getitem__ makes it iterable."""
    
    def __init__(self, *items):
        self._items = list(items)
    
    def __getitem__(self, index):
        return self._items[index]  # Raises IndexError → stops iteration

seq = OldStyleSequence("a", "b", "c")

# Works in for loop! Python calls seq[0], seq[1], ... until IndexError
for item in seq:
    print(f"  {item}", end=" ")
print()

# Also works with 'in' operator and list()
print(f"'b' in seq: {'b' in seq}")
print(f"list(seq):  {list(seq)}")

print("\n💡 This is a legacy pattern — prefer __iter__ for new code")

## 6.5 `reversed()` and `__reversed__`

The built-in `reversed()` works on sequences with `__len__` + `__getitem__`,
but you can customize it with `__reversed__`.

In [ ]:
class Playlist:
    """Custom iterable supporting both forward and reverse iteration."""
    
    def __init__(self, songs: list[str]):
        self._songs = songs
    
    def __iter__(self):
        return iter(self._songs)
    
    def __reversed__(self):
        """Custom reverse iteration."""
        return iter(self._songs[::-1])
    
    def __len__(self):
        return len(self._songs)

playlist = Playlist(["Song A", "Song B", "Song C", "Song D"])

print("Forward:")
for song in playlist:
    print(f"  ▶ {song}")

print("\nReversed:")
for song in reversed(playlist):
    print(f"  ◀ {song}")

## 6.6 Real-World: Paginated API Iterator

This is where class-based iterators shine — managing state across pages.

In [ ]:
class PaginatedAPI:
    """Iterator that fetches data page by page.
    Real-world use: paginated REST APIs, database cursors."""
    
    def __init__(self, total_items: int, page_size: int = 3):
        self._total = total_items
        self._page_size = page_size
        self._offset = 0
    
    def __iter__(self):
        self._offset = 0  # Reset for reuse
        return self
    
    def __next__(self) -> list[dict]:
        if self._offset >= self._total:
            raise StopIteration
        
        # Simulate API call
        end = min(self._offset + self._page_size, self._total)
        page = [
            {"id": i, "name": f"Item {i}"}
            for i in range(self._offset, end)
        ]
        print(f"  📡 Fetched page (offset={self._offset}, size={len(page)})")
        self._offset = end
        return page

api = PaginatedAPI(total_items=8, page_size=3)

print("Iterating through all pages:")
for page in api:
    print(f"    {page}")

print("\n💡 Use class-based iterators when you need:")
print("   - Complex state (page tracking, auth tokens)")
print("   - Reusable iteration (reset offset)")
print("   - Custom methods (e.g., .skip_page())")

## 6.7 `itertools` — Combinatoric Tools

We covered `chain`, `islice`, `takewhile`, `count`, `groupby` in **notebook 05**.
Here are the **combinatoric** tools unique to `itertools`:

In [ ]:
from itertools import (
    cycle, repeat, accumulate,
    product, combinations, permutations, dropwhile
)

# cycle — repeat an iterable forever
from itertools import islice
colors = cycle(["red", "green", "blue"])
print("cycle:       ", list(islice(colors, 7)))

# repeat — repeat a value N times (or forever)
print("repeat:      ", list(repeat("x", 5)))

# accumulate — running totals
print("accumulate:  ", list(accumulate([1, 2, 3, 4, 5])))  # [1, 3, 6, 10, 15]

# dropwhile — skip items while condition holds
print("dropwhile:   ", list(dropwhile(lambda x: x < 5, [1, 3, 6, 2, 1])))

# product — cartesian product (nested loops)
print("product:     ", list(product("AB", [1, 2])))  # ('A',1), ('A',2), ('B',1), ('B',2)

# combinations — C(n,k)
print("C(3,2):      ", list(combinations([1, 2, 3], 2)))

# permutations — P(n,k)
print("P(3,2):      ", list(permutations([1, 2, 3], 2)))

## 6.8 Iterator vs Generator — When to Use Which

| Feature | Class Iterator | Generator Function |
|---------|---------------|-------------------|
| Syntax | `__iter__` + `__next__` | `yield` |
| State | Explicit attributes | Automatic (local variables) |
| Reusable | Can be (with separate iterable) | No — create a new one |
| Methods | Can add custom methods | Just iteration |
| Best for | Complex state / reusable objects | Simple sequences / pipelines |

In [ ]:
# The same thing — class vs generator

# Class-based (verbose but reusable)
class Squares:
    def __init__(self, n):
        self.n = n
    def __iter__(self):
        self.i = 0
        return self
    def __next__(self):
        if self.i >= self.n:
            raise StopIteration
        result = self.i ** 2
        self.i += 1
        return result

# Generator-based (simpler)
def squares(n):
    for i in range(n):
        yield i ** 2

print(f"Class:     {list(Squares(5))}")
print(f"Generator: {list(squares(5))}")

## 6.9 Exercises

**Exercise 1:** Create a `Fibonacci` class-based iterator that:
- Takes a `limit` parameter (max number of values to produce)
- Implements `__iter__` and `__next__`
- Is **reusable** (can iterate multiple times)

In [ ]:
# Your solution here


In [ ]:
# ✅ Solution
class Fibonacci:
    """Reusable Fibonacci iterator."""
    
    def __init__(self, limit: int):
        self.limit = limit
    
    def __iter__(self):
        self._a, self._b = 0, 1
        self._count = 0
        return self
    
    def __next__(self) -> int:
        if self._count >= self.limit:
            raise StopIteration
        value = self._a
        self._a, self._b = self._b, self._a + self._b
        self._count += 1
        return value

fib = Fibonacci(10)
print(f"First pass:  {list(fib)}")
print(f"Second pass: {list(fib)}")   # Reusable!
print(f"Sum: {sum(fib)}")

**Exercise 2:** Create a `CyclicBuffer` class that:
- Stores a fixed-size buffer of items
- Supports `__iter__` (iterate through current items)
- Supports `__reversed__`
- Has an `add(item)` method

In [ ]:
# Your solution here


In [ ]:
# ✅ Solution
from collections import deque

class CyclicBuffer:
    """Fixed-size buffer that drops oldest items."""
    
    def __init__(self, maxsize: int):
        self._buffer = deque(maxlen=maxsize)
    
    def add(self, item):
        self._buffer.append(item)
    
    def __iter__(self):
        return iter(self._buffer)
    
    def __reversed__(self):
        return reversed(self._buffer)
    
    def __len__(self):
        return len(self._buffer)
    
    def __repr__(self):
        return f"CyclicBuffer({list(self._buffer)})"

buf = CyclicBuffer(maxsize=3)
for x in ["a", "b", "c", "d", "e"]:
    buf.add(x)
    print(f"  add('{x}') → {buf}")

print(f"\nForward:  {list(buf)}")
print(f"Reversed: {list(reversed(buf))}")

## 📋 Takeaways

| # | Concept | Key Point |
|---|---------|----------|
| 1 | Iterable | Has `__iter__()` — can create an iterator |
| 2 | Iterator | Has `__next__()` — produces values until `StopIteration` |
| 3 | `for` loop | Calls `iter()` then `next()` repeatedly |
| 4 | Single-use | `__iter__` returns `self` → single-use |
| 5 | Reusable | Separate iterable/iterator classes → multi-use |
| 6 | `__getitem__` | Legacy fallback — Python calls `[0]`, `[1]`... until `IndexError` |
| 7 | `__reversed__` | Custom `reversed()` support |
| 8 | Paginated iterator | Real-world use: API pages, DB cursors |
| 9 | `itertools` combos | `product`, `combinations`, `permutations`, `cycle` |
| 10 | Generator vs Class | Generator = simpler, Class = reusable + custom methods |
| 11 | Java comparison | `__iter__/__next__` ≈ `Iterable/Iterator` interfaces |